# minibayes Quickstart

A 5-minute introduction to Bayesian inference with minibayes.

This notebook covers:
1. Defining a model (priors + likelihood)
2. Running MCMC sampling
3. Inspecting results
4. Basic visualization

In [ ]:
import numpy as np
import minibayes as mb
from minibayes import Model, dist, viz

## The Problem: Robust Linear Regression

We have data with an outlier. A standard Normal likelihood would let the outlier dominate the fit. Instead, we use a Student-t likelihood with heavier tails to downweight extreme observations.

**Model:**
$$y_i = \alpha + \beta x_i + \epsilon_i, \quad \epsilon_i \sim \text{StudentT}(\nu=4, 0, \sigma)$$

**Priors:**
- $\alpha \sim \text{Normal}(0, 10)$
- $\beta \sim \text{Normal}(0, 10)$  
- $\sigma \sim \text{HalfNormal}(5)$

In [ ]:
# Generate data with one outlier
np.random.seed(42)

true_alpha = 1.5
true_beta = 2.0
true_sigma = 0.3

x = np.linspace(0, 4.5, 10)
y = true_alpha + true_beta * x + np.random.normal(0, true_sigma, len(x))

# Add an outlier
y[7] += 4.0

print(f"Data: {len(x)} points")
print(f"True parameters: alpha={true_alpha}, beta={true_beta}, sigma={true_sigma}")

In [ ]:
# Plot the data
with viz.style():
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(x, y, s=60, color=viz.PALETTE["terracotta"], label="Data", zorder=3)
    ax.scatter(x[7], y[7], s=100, color=viz.PALETTE["blue"], marker="x",
               linewidths=3, label="Outlier", zorder=4)
    ax.plot(x, true_alpha + true_beta * x, "--", color=viz.PALETTE["sage"],
            lw=2, label="True line")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    ax.set_title("Linear Regression Data (with outlier)")
    plt.show()

## Step 1: Define the Model

A minibayes model has two parts:
- **priors**: a function that registers parameters using `p(name, distribution)`
- **log_likelihood**: a function that returns the log probability of the data given parameters

In [ ]:
def priors(p):
    """Register model parameters with their prior distributions."""
    p("alpha", dist.Normal(0, 10))      # intercept
    p("beta", dist.Normal(0, 10))       # slope
    p("sigma", dist.HalfNormal(5))      # noise scale (must be positive)


def log_likelihood(params, data):
    """Compute log probability of data given parameters."""
    x_data, y_data = data

    # Predicted mean
    mu = params["alpha"] + params["beta"] * x_data

    # Student-t likelihood (df=4 gives heavier tails than Normal)
    return dist.StudentT(df=4, loc=mu, scale=params["sigma"]).log_prob(y_data)


# Create the model
model = Model(priors=priors, log_likelihood=log_likelihood)

print(f"Parameters: {model.param_names}")
print(f"Transforms: {model.transforms}")

Note that `sigma` automatically gets a `LogTransform` because `HalfNormal` has positive support. The sampler works in unconstrained space, and minibayes handles the transformation.

## Step 2: Run MCMC Sampling

The `sample()` function runs Markov Chain Monte Carlo to draw samples from the posterior distribution.

In [ ]:
result = mb.sample(
    model,
    data=(x, y),
    num_samples=5000,
    num_warmup=1000,
    num_chains=2,
    sampler="adaptive_mh",  # automatically tunes proposal covariance
    seed=42,
    progress=True,
)

print(f"\nSampling completed in {result.elapsed_time:.2f}s")
print(f"Acceptance rate: {result.acceptance_rate.mean():.1%}")

## Step 3: Inspect Results

The `summary()` method computes posterior statistics including effective sample size (ESS) and R-hat convergence diagnostic.

In [ ]:
# Print summary table
print(viz.summary_table(result))

In [ ]:
# Compare to true values
summary = result.summary()

print("\nComparison with true values:")
print(f"  alpha: estimated {summary['alpha']['mean']:.3f} (true: {true_alpha})")
print(f"  beta:  estimated {summary['beta']['mean']:.3f} (true: {true_beta})")
print(f"  sigma: estimated {summary['sigma']['mean']:.3f} (true: {true_sigma})")

## Step 4: Visualization

minibayes includes built-in plotting functions for common diagnostics.

In [ ]:
# Posterior density plots
fig = viz.plot_density(result)
plt.show()

In [ ]:
# Trace plots (samples over iteration)
fig = viz.plot_samples(result)
plt.show()

In [ ]:
# Forest plot (credible intervals)
fig = viz.plot_forest(result)
plt.show()

### Prior vs Posterior Comparison

Compare how the data updated our beliefs from the prior to the posterior.

In [ ]:
# Compare prior vs posterior for alpha
alpha_prior = model.param_info["alpha"].distribution
fig = viz.plot_prior_posterior(alpha_prior, result.samples["alpha"], parameter_name="alpha")
plt.show()

### Posterior Predictive Check

Compare the observed data distribution against simulated data from the prior and posterior.

In [ ]:
# Generate prior and posterior predictive samples for y
# Sample across a wider x-range for a proper marginal PPC
x_ppc = np.linspace(0, 5, 50)  # 50 points across the x range

def predictive_y(params, rng):
    mu = params["alpha"] + params["beta"] * x_ppc
    return {"y": dist.StudentT(df=4, loc=mu, scale=params["sigma"]).sample(size=len(x_ppc), rng=rng)}

prior_pred = mb.sample_prior_predictive(model, predictive_y, num_samples=1000, seed=42)
post_pred = result.predict(predictive_y, num_samples=1000, seed=42)

# Compare observed vs predicted
fig = viz.plot_ppc(y, post_pred["y"].flatten(), prior_predictive=prior_pred["y"].flatten(), bins=100, xlim=[-25, 25])
plt.show()

## Posterior Predictive

Generate predictions from the posterior to visualize uncertainty in the fitted line.

In [ ]:
# Prediction grid
x_plot = np.linspace(-0.5, 5, 100)

def predict_mean(params, rng):
    """Predict the mean (no observation noise)."""
    return {"y": params["alpha"] + params["beta"] * x_plot}

# Generate predictions for each posterior sample
ppc = result.predict(predict_mean, num_samples=500, seed=42)

print(f"Predictions shape: {ppc['y'].shape}  # (num_samples, num_x_points)")

In [ ]:
# Plot posterior predictive with uncertainty band
fig = viz.plot_predictive(x_plot, ppc["y"], x_obs=x, y_obs=y, credible_interval=0.9)
ax = fig.get_axes()[0]

# Add true line for comparison
ax.plot(x_plot, true_alpha + true_beta * x_plot, "--",
        color=viz.PALETTE["sage"], lw=2, label="True line")
ax.legend()
ax.set_title("Posterior Predictive (90% credible interval)")
plt.show()

The Student-t likelihood successfully downweights the outlier. The fitted line stays close to the true relationship despite the extreme observation.

## Saving Results

Results can be saved and loaded for later use.

In [ ]:
# Save to file
# result.save("quickstart_posterior.npz")

# Load later
# loaded = mb.InferenceResult.load("quickstart_posterior.npz")